In [ ]:

import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


# 1. Bootstrap Sample
def bootstrap_sample(n, seed):

    rng = np.random.RandomState(seed)

    idx_in = rng.choice(n, size=n, replace=True)

    all_indices = np.arange(n)

    idx_oob = np.setdiff1d(all_indices, np.unique(idx_in))

    return idx_in, idx_oob


# 2. Single Tree Accuracy
def single_tree_accuracy(X_train, y_train, X_test, y_test, seed=0):

    model = DecisionTreeClassifier(random_state=seed)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    return round(float(accuracy_score(y_test, y_pred)), 4)


# 3. Bagged Accuracy
def bagged_accuracy(
    X_train, y_train, X_test, y_test,
    n_trees=25, seed=0
):

    predictions = []

    n = len(X_train)

    for b in range(n_trees):

        idx_in, idx_oob = bootstrap_sample(n, seed + b)

        X_bootstrap = X_train[idx_in]
        y_bootstrap = y_train[idx_in]

        model = DecisionTreeClassifier(random_state=seed + b)

        model.fit(X_bootstrap, y_bootstrap)

        pred = model.predict(X_test)

        predictions.append(pred)

    predictions = np.array(predictions)

    # Majority voting
    final_predictions = (
        np.sum(predictions, axis=0) >= (n_trees / 2)
    ).astype(int)

    return round(
        float(accuracy_score(y_test, final_predictions)), 4
    )


# 4. Out-of-Bag Score
def oob_score(X, y, n_trees=50, seed=0):

    n = len(X)

    votes = np.zeros((n, 2), dtype=int)

    for b in range(n_trees):

        idx_in, idx_oob = bootstrap_sample(n, seed + b)

        X_bootstrap = X[idx_in]
        y_bootstrap = y[idx_in]

        model = DecisionTreeClassifier(random_state=seed + b)

        model.fit(X_bootstrap, y_bootstrap)

        if len(idx_oob) == 0:
            continue

        predictions = model.predict(X[idx_oob])

        for index, prediction in zip(idx_oob, predictions):
            votes[index, prediction] += 1

    # Samples receiving at least one OOB vote
    valid_mask = votes.sum(axis=1) > 0

    # Majority voting
    final_predictions = np.argmax(votes[valid_mask], axis=1)

    score = accuracy_score(y[valid_mask], final_predictions)

    return round(float(score), 4)


# Verification
d = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    d.data,
    d.target,
    test_size=0.25,
    stratify=d.target,
    random_state=7
)


# Bootstrap verification
idx_in, idx_oob = bootstrap_sample(100, 0)

print("In-bag samples:", len(idx_in))
print("OOB samples:", len(idx_oob))

print("\nSingle Tree Accuracy:")
st = single_tree_accuracy(
    X_train, y_train, X_test, y_test, 0
)
print(st)


print("\nBagged Accuracy:")
bg = bagged_accuracy(
    X_train, y_train, X_test, y_test, 25, 0
)
print(bg)


print("\nOOB Score:")
print(oob_score(d.data, d.target, 50, 0))
